# First Prototype

The starting point of the `ssp` series: a bare **single-series local-level Kalman filter**, written
from scratch and driven by NUTS.

The observation is `log(sales)` from `resource/seasonal`. A day-of-week + yearly Fourier block
(`x_seas`) is regressed out linearly, and what remains is modelled as a random-walk level

$$y_t = a_t + x_t^\top\beta + \varepsilon_t,\quad \varepsilon_t \sim N(0, \sigma_h^2),
\qquad a_{t+1} = a_t + \eta_t,\quad \eta_t \sim N(0, \sigma_q^2).$$

`kalman_filter` returns the exact marginal log-likelihood of $(\sigma_h, \sigma_q, \beta)$ with the
level integrated out, which is handed to NUTS as a `numpyro.factor`. Everything later in the series --
time-point priors (`ssp_02`), the EKF (`ssp_03`), multiple series (`ssp_04` / `ssp_05`) -- is built on
this loop.


In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax.numpy as jnp
from jax import lax

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from typing import Union, Tuple
from jax import random

from bunobee.regression import make_fourier_series_with_ts, make_peridoic_dummies

df = pd.read_csv('../resource/seasonal/df.csv', parse_dates=['date'])
df

In [ ]:
response = df["sales"].values
# response_norm = response.mean()
response_norm = 1
y = np.log(response / response_norm)
y = jnp.array(y)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="Log Sales")
ax.set_title("Log sales")
ax.legend()


In [ ]:
# Seasonality design consumed by `_nuts_fn` below. The latent level `a_t` already carries the
# intercept, so drop the first weekly dummy to keep the design identifiable against it.
x_seas = np.concatenate(
    [
        # (n_steps, 6) day-of-week
        make_peridoic_dummies(len(y), period=7, drop_first=True),
        # (n_steps, 6) yearly, order-3 Fourier
        make_fourier_series_with_ts(df["date"], period=365.25, order=3),
    ],
    axis=1,
)
x_seas = jnp.array(x_seas, dtype=jnp.float32)
x_seas.shape


In [ ]:
def kalman_filter(
    # (1, )
    a0: jnp.ndarray,
    # (1, )
    P0: jnp.ndarray,
    # (1, )
    sigma_h: Union[jnp.ndarray, float],
    # (1, )
    sigma_q: Union[jnp.array, float],
    # (n_steps, )
    y: jnp.array,
    logp: bool = False,
) -> Tuple[float, jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.array, jnp.ndarray]:
    sigma_h_sq = jnp.square(sigma_h)
    sigma_q_sq = jnp.square(sigma_q)

    def _transition_fn(carry, t):
        at, Pt, log_p = carry
        yt = y[t]
        vt = yt - at
        Ft = Pt + sigma_h_sq
        Kt = Pt / Ft

        if logp:
            log_p += -0.5 * jnp.sum(jnp.log(2 * jnp.pi) + jnp.log(Ft) + jnp.square(vt) / Ft)

        # (n_series, n_regressors)
        at = at + Kt * vt

        # (n_series, n_regressors)
        Pt = Pt * (1 -  Kt) + sigma_q_sq

        return ((at, Pt, log_p), (at, Pt, vt, Ft, Kt))

    (_, _, log_p), (at, Pt, vt, Ft, Kt) = lax.scan(
        _transition_fn,
        (a0, P0, 0.0),
        jnp.arange(y.shape[0]),
        length=y.shape[0],
    )
    return log_p, at, Pt, vt, Ft, Kt


def kalman_smooth(
    a0: jnp.ndarray,
    P0: jnp.ndarray,
    sigma_h: Union[jnp.ndarray, float],
    sigma_q: Union[jnp.ndarray, float],
    obs: jnp.ndarray,
    rng_key: jnp.ndarray,
) -> jnp.ndarray:

    # step 1. simulate obs based on non-optimized alpha
    # step 2. run kalman smoother backward
    # step 3. run kalman filter forward
    # step 4. derive smoothed alpha for full dist. of alpha

    n_steps = obs.shape[0]
    sigma_q_sq = jnp.square(sigma_q)

    rng_sub_keys = random.split(rng_key, 3)
    # note that this alpha is ~ N(0, P0) not N(a0, P0)
    init_sim_alpha = dist.Normal(0.0, 1.0).sample(
        rng_sub_keys[0],
    ) * jnp.sqrt(P0)

    obs_noise = dist.Normal(0.0, 1.0).sample(
        rng_sub_keys[1],
        sample_shape=(n_steps, ),
    ) * sigma_h

    innov = dist.Normal(0.0, 1.0).sample(
        rng_sub_keys[2],
        sample_shape=(n_steps, ),
    ) * sigma_q

    def sim_obs_fn(carry, t):
        alpha_t = carry

        # observations simulate step
        y_t = alpha_t + obs_noise[t]
        alpha_t = alpha_t + innov[t]
        return alpha_t, (alpha_t, y_t)

    _, (alpha_plus, obs_plus) = lax.scan(
        sim_obs_fn,
        init=init_sim_alpha,
        xs=jnp.arange(n_steps),
        length=n_steps,
    )

    # output from scan has an additional shape in last dim
    obs_diff = obs - jnp.squeeze(obs_plus, -1)

    _, _, _, v, F, K = kalman_filter(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=obs_diff,
    )

    def kalman_smoother_backward_fn(carry, t):
        rt = carry
        Lt = 1 - K[t]
        rt = 1 / F[t] * v[t] + Lt * rt

        return rt, rt

    rT = jnp.zeros((1))
    _, r = lax.scan(
        kalman_smoother_backward_fn,
        init=rT,
        xs=jnp.arange(n_steps - 1, -1, -1),
        length=n_steps,
    )

    r = jnp.squeeze(r, -1)
    # flip on steps
    r = jnp.flip(r, -1)
    # include first step
    r = jnp.concatenate([r, rT], -1)

    def kalman_smoother_forward_fn(carry, t):
        alpha_t = carry
        # correction of mean vector given lookahead data points
        # note that r vector is appended with initial value;
        # so t + 1 means t in the actual maths
        alpha_t = alpha_t + sigma_q_sq * r[t + 1]

        return alpha_t, alpha_t

    alpha_0 = a0 + P0 * r[0]

    _, smoothed_obs_diff_alpha = lax.scan(
        kalman_smoother_forward_fn,
        init=alpha_0,
        xs=jnp.arange(n_steps),
        length=n_steps,
    )   

    smoothed_alpha = alpha_plus + smoothed_obs_diff_alpha
    # (n_steps, )
    return jnp.squeeze(smoothed_alpha, -1)


def simulate_forecast(
    # (n_sample, )
    a0: jnp.ndarray,
    # (n_sample, )
    sigma_h: jnp.ndarray,
    # (n_sample, )
    sigma_q: jnp.ndarray,
    # forecast horizon
    n_steps: int,
    rng_key: jnp.ndarray,
):
    rng_sub_keys = random.split(rng_key, 2)
    n_samples = sigma_h.shape[0]

    # move steps to first dim to facilitate broadcast
    # (n_steps, n_samples)
    obs_noise = (
        dist.Normal(loc=0.0, scale=1.0).sample(
            rng_sub_keys[0], sample_shape=(n_steps, n_samples)
        ) * sigma_h
    )
    innov = (
        dist.Normal(loc=0.0, scale=1.0).sample(
            rng_sub_keys[1], sample_shape=(n_steps, n_samples)
        ) * sigma_q
    )

    def sim_transition_fn(carry, t):
        at = carry
        at = at + innov[t]

        return at, at

    _, states = lax.scan(
        sim_transition_fn,
        a0,
        jnp.arange(n_steps),
        length=n_steps,
    )
    # (n_steps, n_samples) -> (n_samples, n_steps)
    res = states + obs_noise
    res = jnp.swapaxes(res, -1, -2)
    return res

In [ ]:
a0 = jnp.zeros(1)
P0 = jnp.ones(1)

In [ ]:
def _nuts_fn(a0, P0):
    # observation noise
    sigma_h = numpyro.sample(
        'sigma_h',
        dist.TruncatedNormal(
            0, 0.01, high=0.8, low=1e-5
        )
    )
    # state (level) innovation
    sigma_q = numpyro.sample(
        'sigma_q',
        dist.TruncatedNormal(
            0, 0.1, high=0.1, low=1e-5
        )
    )

    beta_seas = numpyro.sample("beta_seas", dist.Normal(0, 0.5).expand([x_seas.shape[1]]))
    reg_comp = jnp.matmul(x_seas, beta_seas)

    lp, at, Pt, _, _, _ = kalman_filter(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y - reg_comp,
        logp=True,
    )

    numpyro.factor("lp", lp)
    numpyro.deterministic("a0", a0)
    numpyro.deterministic("P0", P0)
    numpyro.deterministic("at", jnp.squeeze(at, -1))
    numpyro.deterministic("Pt", Pt)
    numpyro.deterministic("mu", jnp.squeeze(at, -1) + reg_comp)


## Fit

4 chains x (500 warmup + 500 draws). The level is integrated out analytically, so NUTS only explores
`sigma_h`, `sigma_q` and the seasonality coefficients -- a handful of seconds on CPU.


In [ ]:
nuts_kernel = NUTS(_nuts_fn)
mcmc = MCMC(
    nuts_kernel, 
    num_warmup=500, 
    num_samples=500,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key, a0, P0)

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
# at = np.squeeze(posteriors_dict["at"], -1)
# mu_samples = np.array(at)
mu_samples = np.array(posteriors_dict["mu"])
sigma_samples = np.array(posteriors_dict["sigma_h"])

# generate error
eps_samples = np.transpose(
    np.random.normal(loc=0.0, scale=sigma_samples, size=(len(y), sigma_samples.shape[0])),
    axes=(1, 0)
)
print(eps_samples.shape)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), response, label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(len(y)), yhat_mid, label='Filtered fit', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='90% Interval', color="dodgerblue")
ax.set_title('State Space Model, In-Sample Fit')
ax.legend()
